# Phase 1c — LoRA sequence encoder on Colab

Trains **ChemBERTa + LoRA** and its **frozen control** across 8 datasets x 6 scaffold
splits, then brings the results back.

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**. Then run the cells
in order. Expect roughly **1.5-2.5 hours** on a free T4.

**If Colab disconnects** (free sessions do): reconnect, re-run cells 1-5, and cell 6
picks up where it stopped. Nothing is lost -- results are written straight to your
Drive after each split, not held until the end.


## 1. Check you actually got a GPU

Free Colab sometimes hands out a CPU-only runtime at busy times. If this says
`cuda available: False`, the run would still work but take ~20 hours instead of ~2.
Change the runtime type, or come back later.


In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
else:
    print('NO GPU -- Runtime > Change runtime type > T4 GPU, then re-run this cell.')


## 2. Connect your Google Drive

Results are written here as they are produced, so a disconnect costs at most the
split that was in progress.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Point at the bundle

Upload `mpp_colab_bundle.zip` (made by `python -m scripts.make_colab_bundle`) to your
Drive, then set the path below to wherever you put it.


In [ ]:
BUNDLE = '/content/drive/MyDrive/mpp_colab_bundle.zip'  # <- edit if you put it elsewhere
OUTDIR = '/content/drive/MyDrive/mpp_phase1c'           # results land here

import os
assert os.path.exists(BUNDLE), f'Not found: {BUNDLE}  -- check the path and re-run.'
os.makedirs(OUTDIR, exist_ok=True)
print('bundle:', round(os.path.getsize(BUNDLE)/1e6, 1), 'MB')


## 4. Unpack and install

Colab already has PyTorch and transformers. Only `peft` (the LoRA implementation) and
`accelerate` need adding.


In [ ]:
import zipfile, os

WORK = '/content/mpp'
os.makedirs(WORK, exist_ok=True)
with zipfile.ZipFile(BUNDLE) as z:
    z.extractall(WORK)
os.chdir(WORK)

!pip -q install peft accelerate

import subprocess, sys
print(subprocess.run([sys.executable, '-c', 'import peft; print("peft", peft.__version__)'],
                     capture_output=True, text=True).stdout)


## 5. Keep results on Drive, not on the disposable machine

`results/runs/` is where each split's metrics are archived, and it is what the resume
check reads. Pointing it at Drive means an interrupted session resumes correctly and
finished work is never lost with the runtime.


In [ ]:
import os

os.makedirs(f'{OUTDIR}/runs', exist_ok=True)
os.makedirs('results', exist_ok=True)
if not os.path.islink('results/runs'):
    if os.path.exists('results/runs'):
        import shutil; shutil.rmtree('results/runs')
    os.symlink(f'{OUTDIR}/runs', 'results/runs')
print('results/runs ->', os.path.realpath('results/runs'))


## 6. Train

Two encoders across six splits:

- **`lora`** — ChemBERTa with LoRA adapters, CLS + mean pooling
- **`seq_frozen`** — the same encoder with no adaptation, as a matched control. Without
  it, a win for `lora` could equally be the pooling change rather than the fine-tuning.

`--resume` skips any split already finished, so re-running this cell after a disconnect
continues rather than restarting. Progress prints per split; it is safe to leave.

`--artifacts tok ecfp` materialises only what the sequence view reads, which is why the
bundle did not need the graph pools.


In [ ]:
!python -m scripts.run_view_multiseed \
    --tags lora seq_frozen \
    --variants deepchem seed0 seed1 seed2 seed3 seed4 \
    --artifacts tok ecfp \
    --device cuda \
    --resume \
    --restore none


## 7. Check what finished

Each split should have 32 files: 8 datasets x 2 tags x (valid + test).


In [ ]:
import os, glob
for v in ['deepchem','seed0','seed1','seed2','seed3','seed4']:
    n = len(glob.glob(f'{OUTDIR}/runs/{v}/metrics/*_lora_*.csv'))
    m = len(glob.glob(f'{OUTDIR}/runs/{v}/metrics/*_seq_frozen_*.csv'))
    flag = 'ok' if (n == 16 and m == 16) else 'INCOMPLETE - re-run cell 6'
    print(f'{v:<10} lora {n:>2}/16   seq_frozen {m:>2}/16   {flag}')


## 8. Bring the results home

Download this zip and unpack it over your local `results/runs/`, then run the paired
statistics locally:

```bash
python -m src.eval.view_stats --a lora --b seq_frozen
```

Do **not** read the single-split numbers as a result. The Phase 1a lesson was that the
DeepChem split alone reversed direction on 3 of 5 datasets; only the five seeded splits
with intervals settle anything.


In [ ]:
import shutil
out = shutil.make_archive('/content/phase1c_results', 'zip', f'{OUTDIR}/runs')
print('wrote', out, round(os.path.getsize(out)/1e6, 2), 'MB')
from google.colab import files
files.download(out)
